# Surface feature extraction

This file performs minimal cleaning and then passes the texts through the surface pipeline to extract the related surface-level features for each document.

In [8]:
import re
import numpy as np
import pandas as pd
import nltk
from nltk.tokenize import word_tokenize
from nltk.tokenize import sent_tokenize
from collections import Counter

In [9]:
def surface_pipe_clean(text):

    # if text is missing return a blank ""
    if pd.isna(text):
        return ""

    # convert text to string if not already
    text = str(text)

    # remove any html markup still existing
    text = re.sub(r"<[^>]+>", " ", text)

    # change any multiple spaces, tabs, newlines with " "
    text = re.sub(r"\s+", " ", text)

    # strip outside whitespace
    text = text.strip()

    # return cleaned text
    return text

Next we pre-define function words as per the 70 stated in "Stylometric detection of AI-generated texts: evidence from human and machine-written essays" by Chen R. et al. (2026) p2.

In [1]:
defined_function_words = [

    "a", "all", "also", "an", "and", "any", "are", "as", "at", "be", "been", "but", "by", "can", "do", "down", "even", "every", "for",
    "from", "had", "has", "have", "her", "his", "if", "in", "into", "is", "it", "its", "may", "more", "must", "my", "no", "not", "now",
    "of", "on", "one", "only", "or", "our", "shall", "should", "so", "some", "such", "than", "that", "the", "their", "then", "there",
    "things", "this", "to", "up", "upon", "was", "were", "what", "when", "which", "who", "will", "with", "would", "your"

]

One of the features determined is MTLD which is defined in "MTLD, vocd-D, and HD-D: A validation study of sophisticated approaches to lexical diversity assessment" by McCarthy and Jarvis (2010) p384. This needs multiple calculations so wrapped up in this function. The paper suggests the threshold of 0.72.

In [11]:
def mtld_calc(tokens, threshold = 0.72):

    # return 0 if no tokens
    if len(tokens) == 0:
        return 0

    # initalise empty set to store unique words
    types = set()

    # initialise tokens in current factor
    factor_tokens = 0
    # and total factors
    factors = 0

    # now loop through tokens
    for token in tokens:

        # add token to set
        types.add(token)
        # incremenet count
        factor_tokens += 1
        # and calcualte current TTR
        current_ttr = len(types) / factor_tokens

        # finally check whether threshold has been reached
        if current_ttr <= threshold:

            # increment factors
            factors += 1
            # reset set of unique words
            types = set()
            # and reset the factor tokens
            factor_tokens = 0

    # after final factor check to see if there are tokens remaining
    if factor_tokens > 0:

        # get TTR of final partial factor
        current_ttr = len(types) / factor_tokens
        # calculate fraction of final fractor
        final_factor = ((1 - current_ttr) / (1 - threshold))
        # add the partial factor to the total
        factors += final_factor

    # get final mtld if not 0
    if factors > 0:
        
        mtld = len(tokens) / factors

    else:

        mtld = 0
        
    # return mtld score
    return mtld   

Now we need the actual pipline to utilise the functions and word list above to collect all the surface data.

In [12]:
def extract_surface_features(text):

    # run cleaning function
    text = surface_pipe_clean(text)
    # return empty dict if no text
    if not text:
        return {}
    
    ##################################
    # Words
    ##################################

    # tokenize text into words and punct
    all_tokens = word_tokenize(text)
    # trim to words only
    words = [
        token for token in all_tokens 
        if token.isalpha()
    ]
    # make all words lowercase to not duplicate the and The etc.
    words = [
        word.lower() 
        for word in words
    ]
    # finally count tot of all words
    word_count = len(words)
    # also count all chars inc. punct and " "
    # whilst here, needed later for ratios
    char_count = len(text)
    # get length of all words
    word_lengths = [
        len(word) for word in words
    ]
    # get average word length
    mean_word_length = (
        np.mean(word_lengths) 
        if word_lengths 
        else 0
    )
    # get STD of word lengths
    std_word_length = (
        np.std(word_lengths) 
        if word_lengths 
        else 0
    )


    ##################################
    # Sentences
    ##################################

    # tokenise sentences
    sentences = sent_tokenize(text)
    # count them
    sentence_count = len(sentences)
    # get num of words in each sentence
    sentence_lengths = [
        len([
            token for token in word_tokenize(sentence)
            if token.isalpha()
        ])
        for sentence in sentences
    ]
    # Get average sentence length
    mean_sentence_length = (
        np.mean(sentence_lengths) 
        if sentence_lengths 
        else 0
    )
    # get STD of sentence lengths
    std_sentence_length = (
        np.std(sentence_lengths) 
        if sentence_lengths 
        else 0
    )


    ##################################
    # Word frequencies
    ##################################

    # These are used in the calculations for  ratios etc.
    # get word counts of every word
    word_counts = Counter(words)
    # calculate num of different words used
    vocab_size = len(word_counts)


    ##################################
    # Ratios
    ##################################

    # calculate Root of type token ratio (TTR)
    root_ttr = (
        vocab_size / np.sqrt(word_count)
        if word_count > 0 
        else 0
    )
    # hapax legomena count - words occuring only once
    hapax_l_count = sum(
        1 for frequency in word_counts.values()
        if frequency == 1
    )
    # hapax l ratio (hapax words / all)
    hapax_l_ratio = (
        hapax_l_count / word_count
        if word_count > 0 
        else 0
    )
    # calculate hapax dislegomena count - words appearing twice
    hapax_d_count = sum(
        1 for frequency in word_counts.values()
        if frequency == 2
    )
    # hapax d ratio (hapax words / all)
    hapax_d_ratio = (
        hapax_d_count / word_count
        if word_count > 0 
        else 0
    )
    # calculate Yuke's K - vocab repetition
    # get sum of squared word freq
    sum_square_freq = sum(
        freq ** 2
        for freq in word_counts.values()
    )
    # calculate Yules K measure
    yules_k = (
        10000 * (sum_square_freq - word_count) / (word_count ** 2)
        if word_count > 0 
        else 0
    )
    # calculate mtld
    mtld = mtld_calc(words)


    ##################################
    # Function word frequencies
    ##################################

    # initialise empty dict
    function_word_ratios = {}
    # loop through pre-defined function words
    for function_word in defined_function_words:
        # assign f sting name
        function_word_ratios[f"func_ratio_{function_word}"] = (
            # assign value of function word count / total
            word_counts[function_word] / word_count
            if word_count > 0 
            else 0
        )


    ##################################
    # Punctation
    ##################################

    # counts of all common punctuation types
    comma_count =       text.count(",")
    colon_count =       text.count(":")
    semicolon_count =   text.count(";")
    question_count =    text.count("?")
    exclamation_count = text.count("!")
    hyphen_count =      text.count("-")
    endash_count =      text.count("–")
    emdash_count =      text.count("—")

    # ratios for all the above / total characters in text
    comma_ratio = (
        comma_count / char_count 
        if char_count > 0 
        else 0
    )
    colon_ratio = (
        colon_count / char_count 
        if char_count > 0 
        else 0
    )
    semicolon_ratio = (
        semicolon_count / char_count 
        if char_count > 0 
        else 0
    )
    question_ratio = (
        question_count / char_count 
        if char_count > 0 
        else 0
    )
    exclamation_ratio = (
        exclamation_count / char_count 
        if char_count > 0 
        else 0
    )
    hyphen_ratio = (
        hyphen_count / char_count 
        if char_count > 0 
        else 0
    )
    endash_ratio = (
        endash_count / char_count 
        if char_count > 0 
        else 0
    )
    emdash_ratio = (
        emdash_count / char_count 
        if char_count > 0 
        else 0
    )


    ##################################
    # Character types
    ##################################

    # count alphabetic
    alpha_count = sum(
        1 for character in text
        if character.isalpha()
    )
    # count uppercase
    upper_count = sum(
        1 for character in text
        if character.isupper()
    )
    # count lowercase
    lower_count = sum(
        1 for character in text
        if character.islower()
    )
    # count digits
    digit_count = sum(
        1 for character in text
        if character.isdigit()
    )
    # count whitespaces
    space_count = sum(
        1 for character in text
        if character.isspace()
    )
    # count punctuation, i.e. not alphanumeric or " "
    punct_count = sum(
        1 for character in text
        if not character.isalnum()
        and not character.isspace()
    )
    
    # as above, ratios for all the above / total chars in text
    alpha_ratio = (
        alpha_count / char_count 
        if char_count > 0 
        else 0
    )
    upper_ratio = (
        upper_count / char_count 
        if char_count > 0 
        else 0
    )
    lower_ratio = (
        lower_count / char_count 
        if char_count > 0 
        else 0
    )
    digit_ratio = (
        digit_count / char_count 
        if char_count > 0 
        else 0
    )
    space_ratio = (
        space_count / char_count 
        if char_count > 0 
        else 0
    )
    punct_ratio = (
        punct_count / char_count 
        if char_count > 0 
        else 0
    )


    ##################################
    # Storage
    ##################################

    # now need to store all variables gathered in the pipeline
    surface_features = {

        # Totals
        "word_count" :           word_count,
        "character_count" :      char_count,
        "sentence_count" :       sentence_count,
        
        # Words
        "mean_word_length" :     mean_word_length,
        "std_word_length" :      std_word_length,

        # Sentences
        "mean_sentence_length" : mean_sentence_length,
        "std_sentence_length" :  std_sentence_length,

        # Lexical diversity features
        "root_ttr" :             root_ttr,
        "hapax_l_ratio" :        hapax_l_ratio,
        "hapax_d_ratio" :        hapax_d_ratio,
        "yules_k" :              yules_k,
        "mtld" :                 mtld,

        # Punctuation
        "comma_ratio" :          comma_ratio,
        "colon_ratio" :          colon_ratio,
        "semicolon_ratio" :      semicolon_ratio,
        "question_ratio" :       question_ratio,
        "exclamation_ratio" :    exclamation_ratio,
        "hyphen_ratio" :         hyphen_ratio,
        "endash_ratio" :         endash_ratio,
        "emdash_ratio" :         emdash_ratio,

        # Characters
        "alpha_ratio" :          alpha_ratio,
        "upper_ratio" :          upper_ratio,
        "lower_ratio" :          lower_ratio,
        "digit_ratio" :          digit_ratio,
        "space_ratio" :          space_ratio,
        "punct_ratio" :          punct_ratio

    }
    
    # Function words
    # Finally add on all the function word ratios
    surface_features.update(function_word_ratios)

    # ...and return all features
    return surface_features

Pipeline should be working now, lets check on a piece of text.

In [6]:
# This is copied from today's BBC news @ https://www.bbc.co.uk/news/articles/cn9wv7pene2o

test_text = """

Conservative leader Kemi Badenoch has pressed Andy Burnham over UK borrowing and spending at his first Prime Minister's Questions.
Badenoch called on the PM to say where spending would be cut and how he would deal with the UK's increasing debt, after the cost of 
borrowing for the UK rose, reaching a new 18-year high on Wednesday. The PM blamed the previous Conservative government saying that the 
"turbulence on global markets are because of that exposure that they left behind". Burnham became prime minister in July, having taken 
over from Sir Keir Starmer just before Parliament's summer recess, and it was the first time he had faced the Tory leader at the Commons session.
The prime minister used the summer recess to tour the UK, and had announced a series of eye-catching policies, mainly aimed at tackling the 
cost of living. MPs returned to Parliament on Tuesday and Burnham spent more than three hours in the Commons, making a statement and 
answering questions from MPs. At PMQs, Badenoch said the prime minister had set out plenty of spending plans but no detail of how he would be 
paying the bill. "The markets are clearly worried that we now have a spendthrift prime minister who wants to say yes to everyone but cannot 
tell us where the money is coming from," she said. "He needs to pay for all his new promises and he has a choice - higher taxes, more borrowing, 
or spending cuts. "Yesterday he said change begins with honesty, so will he be honest and tell us: is he preparing to raise taxes again, yes or no?"
Burnham had earlier said he had already cut taxes, including VAT on energy bills and a cut in business rates for hospitality. "I've already 
indicated my first moves were to cut tax, but I'm not going to do what any prime minister has done and write the budget here," he said, adding 
he and Chancellor John Healey had agreed on an early date for the Budget in October to reduce speculation. At PMQs, Badenoch also raised comments 
from economist and cross-bench peer Lord Jim O'Neill, who had been tipped to become Burnham's chief economic adviser, but who ruled out joining 
Burnham's government. Speaking to the BBC, Lord O'Neill had praised the PM for "a great first five weeks" which he said had helped lift consumer 
and business confidence; adding the current spike in market interest rates for government debt would force Labour to start "dealing with the triple 
lock" on the state pension, and "excessive" welfare spending. Badenoch pointed to a warning from Lord O'Neill that the prime minister's tone 
during his Tuesday Commons debut would have unsettled investors, although the peer had also said that spike was mostly mirroring US market 
upheaval concerning the Iran conflict.

"""

In [7]:
# run pipeline on test text
test_features = extract_surface_features(test_text)
# print out values
for feature, value in test_features.items():
    print(f"{feature}: {value}")

word_count: 460
character_count: 2713
sentence_count: 15
mean_word_length: 4.6173913043478265
std_word_length: 2.326903554346147
mean_sentence_length: 30.666666666666668
std_sentence_length: 10.011104945120804
root_ttr: 11.05018197764772
hapax_l_ratio: 0.35434782608695653
hapax_d_ratio: 0.08260869565217391
yules_k: 97.25897920604915
mtld: 117.73650609402206
comma_ratio: 0.008477699963140435
colon_ratio: 0.00036859565057132326
semicolon_ratio: 0.00036859565057132326
question_ratio: 0.00036859565057132326
exclamation_ratio: 0.0
hyphen_ratio: 0.001474382602285293
endash_ratio: 0.0
emdash_ratio: 0.0
alpha_ratio: 0.8020641356431994
upper_ratio: 0.03317360855141909
lower_ratio: 0.7688905270917803
digit_ratio: 0.0007371913011426465
space_ratio: 0.17139697751566532
punct_ratio: 0.025801695539992626
func_ratio_a: 0.017391304347826087
func_ratio_all: 0.002173913043478261
func_ratio_also: 0.004347826086956522
func_ratio_an: 0.002173913043478261
func_ratio_and: 0.030434782608695653
func_ratio_any:

____________________

# Main extraction

Now that pipeline works we want to run train text through it to convert each text to a numerical represntation, with the whole train set becoming feature matrix x_train_surface.

In [9]:
# load training data
train_df = pd.read_csv(
    r"C:\Users\Jamie\Desktop\DSM500 Final Project\Code\Final datasets v2 last update\train_dataset_master.csv"
)

# run surface feature extraction on first 20 texts to ensure it works as expected
X_train_surface = pd.DataFrame(
    train_df["text"].iloc[:20].apply(extract_surface_features
    # and convert to list of dicts
    ).to_list()
)

# check output
print(X_train_surface.shape)
print(X_train_surface.isna().sum().sum())
print(X_train_surface.dtypes.value_counts())
X_train_surface.head(10)

(20, 96)
0
float64    93
int64       3
Name: count, dtype: int64


,word_count,character_count,sentence_count,mean_word_length,std_word_length,mean_sentence_length,std_sentence_length,root_ttr,hapax_l_ratio,hapax_d_ratio,...,func_ratio_was,func_ratio_were,func_ratio_what,func_ratio_when,func_ratio_which,func_ratio_who,func_ratio_will,func_ratio_with,func_ratio_would,func_ratio_your
0,373,2722,22,5.994638,3.000442,16.954545,4.958714,12.426729,0.506702,0.085791,...,0.008043,0.018767,0.000000,0.002681,0.000000,0.000000,0.000000,0.008043,0.000000,0.000000
1,373,2284,16,4.949062,2.787386,23.312500,8.371968,11.598280,0.458445,0.080429,...,0.010724,0.005362,0.002681,0.000000,0.002681,0.002681,0.000000,0.002681,0.002681,0.000000
2,368,2394,16,5.027174,2.529032,23.000000,7.271520,12.354479,0.529891,0.054348,...,0.010870,0.002717,0.002717,0.002717,0.002717,0.002717,0.008152,0.002717,0.005435,0.000000
3,419,2606,17,4.952267,2.627669,24.647059,8.567663,12.408712,0.463007,0.083532,...,0.004773,0.007160,0.000000,0.000000,0.002387,0.004773,0.004773,0.004773,0.002387,0.000000
4,402,2126,25,3.947761,1.988714,16.080000,8.643703,10.274346,0.345771,0.089552,...,0.012438,0.000000,0.002488,0.002488,0.002488,0.000000,0.002488,0.004975,0.000000,0.000000
5,383,2473,17,5.007833,2.585850,22.529412,6.722374,9.912937,0.365535,0.062663,...,0.013055,0.000000,0.002611,0.002611,0.005222,0.002611,0.000000,0.005222,0.000000,0.000000
6,458,2505,17,4.176856,2.070100,26.941176,23.304810,11.307917,0.373362,0.069869,...,0.006550,0.004367,0.002183,0.000000,0.000000,0.004367,0.000000,0.006550,0.013100,0.002183
7,381,2112,19,4.440945,2.383188,20.052632,7.380441,11.270941,0.456693,0.057743,...,0.007874,0.002625,0.000000,0.005249,0.002625,0.000000,0.000000,0.010499,0.000000,0.000000
8,385,2350,18,4.612987,2.173832,21.388889,12.499506,10.091014,0.353247,0.072727,...,0.000000,0.000000,0.000000,0.000000,0.002597,0.000000,0.002597,0.007792,0.002597,0.007792
9,431,2401,33,4.296984,1.995346,13.060606,6.362482,11.849403,0.422274,0.081206,...,0.006961,0.002320,0.002320,0.002320,0.002320,0.000000,0.000000,0.004640,0.000000,0.002320


All appears to be working well, no NaNs, shape as expected, primarily floats with 3 intergers - word count, char count and sentence count. Can move to going through entire train dataset now.

In [10]:
# all as above but now across all texts
train_df = pd.read_csv(
    r"C:\Users\Jamie\Desktop\DSM500 Final Project\Code\Final datasets v2 last update\train_dataset_master.csv"
)

# run surface feature extraction across entire train set
X_train_surface = pd.DataFrame(
    train_df["text"].apply(extract_surface_features
    # and convert to list of dicts
    ).to_list()
)

# check output
print(X_train_surface.shape)
print(X_train_surface.isna().sum().sum())
print(X_train_surface.dtypes.value_counts())
X_train_surface.head(10)

(3200, 96)
0
float64    93
int64       3
Name: count, dtype: int64


,word_count,character_count,sentence_count,mean_word_length,std_word_length,mean_sentence_length,std_sentence_length,root_ttr,hapax_l_ratio,hapax_d_ratio,...,func_ratio_was,func_ratio_were,func_ratio_what,func_ratio_when,func_ratio_which,func_ratio_who,func_ratio_will,func_ratio_with,func_ratio_would,func_ratio_your
0,373,2722,22,5.994638,3.000442,16.954545,4.958714,12.426729,0.506702,0.085791,...,0.008043,0.018767,0.000000,0.002681,0.000000,0.000000,0.000000,0.008043,0.000000,0.000000
1,373,2284,16,4.949062,2.787386,23.312500,8.371968,11.598280,0.458445,0.080429,...,0.010724,0.005362,0.002681,0.000000,0.002681,0.002681,0.000000,0.002681,0.002681,0.000000
2,368,2394,16,5.027174,2.529032,23.000000,7.271520,12.354479,0.529891,0.054348,...,0.010870,0.002717,0.002717,0.002717,0.002717,0.002717,0.008152,0.002717,0.005435,0.000000
3,419,2606,17,4.952267,2.627669,24.647059,8.567663,12.408712,0.463007,0.083532,...,0.004773,0.007160,0.000000,0.000000,0.002387,0.004773,0.004773,0.004773,0.002387,0.000000
4,402,2126,25,3.947761,1.988714,16.080000,8.643703,10.274346,0.345771,0.089552,...,0.012438,0.000000,0.002488,0.002488,0.002488,0.000000,0.002488,0.004975,0.000000,0.000000
5,383,2473,17,5.007833,2.585850,22.529412,6.722374,9.912937,0.365535,0.062663,...,0.013055,0.000000,0.002611,0.002611,0.005222,0.002611,0.000000,0.005222,0.000000,0.000000
6,458,2505,17,4.176856,2.070100,26.941176,23.304810,11.307917,0.373362,0.069869,...,0.006550,0.004367,0.002183,0.000000,0.000000,0.004367,0.000000,0.006550,0.013100,0.002183
7,381,2112,19,4.440945,2.383188,20.052632,7.380441,11.270941,0.456693,0.057743,...,0.007874,0.002625,0.000000,0.005249,0.002625,0.000000,0.000000,0.010499,0.000000,0.000000
8,385,2350,18,4.612987,2.173832,21.388889,12.499506,10.091014,0.353247,0.072727,...,0.000000,0.000000,0.000000,0.000000,0.002597,0.000000,0.002597,0.007792,0.002597,0.007792
9,431,2401,33,4.296984,1.995346,13.060606,6.362482,11.849403,0.422274,0.081206,...,0.006961,0.002320,0.002320,0.002320,0.002320,0.000000,0.000000,0.004640,0.000000,0.002320


Have now extracted surface features for all train set. can now save and move onto syntactic.

In [11]:
X_train_surface.to_csv(
    r"C:\Users\Jamie\Desktop\DSM500 Final Project\Code\Final datasets v2 last update\X_train_surface.csv",
    index = False
)

_________________

# Returning from Objective 5

Had to re-run a 300 test set through paraphrasing and so now have a slightly reduced train set of 2900 that must have features extracted again so that training doesn't train on features that are now in the test set. New train set is:

C:\Users\Jamie\Desktop\DSM500 Final Project\Code\Final datasets v2 last update\final_train_2900.csv

Will now re-run on the new train set.

In [6]:
# all copied from above
train_df = pd.read_csv(
    r"C:\Users\Jamie\Desktop\DSM500 Final Project\Code\Final datasets v2 last update\final_train_2900.csv"
)

# run surface feature extraction across entire train set
X_train_surface_2900 = pd.DataFrame(
    train_df["text"].apply(extract_surface_features
    # and convert to list of dicts
    ).to_list()
)

# check output
print(X_train_surface_2900.shape)
print(X_train_surface_2900.isna().sum().sum())
print(X_train_surface_2900.dtypes.value_counts())
X_train_surface_2900.head(10)

(2899, 96)
0
float64    93
int64       3
Name: count, dtype: int64


,word_count,character_count,sentence_count,mean_word_length,std_word_length,mean_sentence_length,std_sentence_length,root_ttr,hapax_l_ratio,hapax_d_ratio,...,func_ratio_was,func_ratio_were,func_ratio_what,func_ratio_when,func_ratio_which,func_ratio_who,func_ratio_will,func_ratio_with,func_ratio_would,func_ratio_your
0,373,2722,22,5.994638,3.000442,16.954545,4.958714,12.426729,0.506702,0.085791,...,0.008043,0.018767,0.000000,0.002681,0.000000,0.000000,0.000000,0.008043,0.000000,0.000000
1,373,2284,16,4.949062,2.787386,23.312500,8.371968,11.598280,0.458445,0.080429,...,0.010724,0.005362,0.002681,0.000000,0.002681,0.002681,0.000000,0.002681,0.002681,0.000000
2,368,2394,16,5.027174,2.529032,23.000000,7.271520,12.354479,0.529891,0.054348,...,0.010870,0.002717,0.002717,0.002717,0.002717,0.002717,0.008152,0.002717,0.005435,0.000000
3,419,2606,17,4.952267,2.627669,24.647059,8.567663,12.408712,0.463007,0.083532,...,0.004773,0.007160,0.000000,0.000000,0.002387,0.004773,0.004773,0.004773,0.002387,0.000000
4,383,2473,17,5.007833,2.585850,22.529412,6.722374,9.912937,0.365535,0.062663,...,0.013055,0.000000,0.002611,0.002611,0.005222,0.002611,0.000000,0.005222,0.000000,0.000000
5,458,2505,17,4.176856,2.070100,26.941176,23.304810,11.307917,0.373362,0.069869,...,0.006550,0.004367,0.002183,0.000000,0.000000,0.004367,0.000000,0.006550,0.013100,0.002183
6,381,2112,19,4.440945,2.383188,20.052632,7.380441,11.270941,0.456693,0.057743,...,0.007874,0.002625,0.000000,0.005249,0.002625,0.000000,0.000000,0.010499,0.000000,0.000000
7,385,2350,18,4.612987,2.173832,21.388889,12.499506,10.091014,0.353247,0.072727,...,0.000000,0.000000,0.000000,0.000000,0.002597,0.000000,0.002597,0.007792,0.002597,0.007792
8,431,2401,33,4.296984,1.995346,13.060606,6.362482,11.849403,0.422274,0.081206,...,0.006961,0.002320,0.002320,0.002320,0.002320,0.000000,0.000000,0.004640,0.000000,0.002320
9,374,2137,18,4.601604,2.422951,20.777778,6.381880,11.582764,0.470588,0.061497,...,0.005348,0.000000,0.002674,0.002674,0.002674,0.000000,0.000000,0.026738,0.005348,0.000000


In [7]:
X_train_surface_2900.to_csv(
    r"C:\Users\Jamie\Desktop\DSM500 Final Project\Code\Final datasets v2 last update\X_train_surface_2900.csv",
    index = False
)

___

And now for the 4 batches of test features.

In [13]:
# all copied from above
train_df = pd.read_csv(
    r"C:\Users\Jamie\Desktop\DSM500 Final Project\Code\Final datasets v2 last update\test_heavy_300.csv"
)

# run surface feature extraction across entire test set
test_heavy_300 = pd.DataFrame(
    train_df["text"].apply(extract_surface_features
    # and convert to list of dicts
    ).to_list()
)

# check output
print(test_heavy_300.shape)
print(test_heavy_300.isna().sum().sum())
print(test_heavy_300.dtypes.value_counts())
test_heavy_300.head(10)

(300, 96)
0
float64    93
int64       3
Name: count, dtype: int64


,word_count,character_count,sentence_count,mean_word_length,std_word_length,mean_sentence_length,std_sentence_length,root_ttr,hapax_l_ratio,hapax_d_ratio,...,func_ratio_was,func_ratio_were,func_ratio_what,func_ratio_when,func_ratio_which,func_ratio_who,func_ratio_will,func_ratio_with,func_ratio_would,func_ratio_your
0,389,2559,17,5.282776,2.617371,22.882353,5.497404,12.168483,0.480720,0.079692,...,0.005141,0.002571,0.000000,0.002571,0.000000,0.002571,0.010283,0.010283,0.002571,0.000000
1,352,2362,20,5.261364,2.736514,17.600000,6.499231,10.180334,0.394886,0.079545,...,0.014205,0.002841,0.000000,0.000000,0.002841,0.000000,0.000000,0.011364,0.000000,0.000000
2,374,2304,18,4.732620,2.674529,20.777778,8.599454,11.944726,0.478610,0.061497,...,0.002674,0.002674,0.008021,0.002674,0.000000,0.002674,0.000000,0.005348,0.010695,0.002674
3,457,3483,24,6.111597,2.928509,19.041667,6.194616,14.033408,0.514223,0.083151,...,0.000000,0.000000,0.000000,0.002188,0.000000,0.000000,0.002188,0.004376,0.000000,0.000000
4,472,3309,22,5.673729,2.889107,21.454545,6.358439,13.716562,0.512712,0.057203,...,0.000000,0.000000,0.000000,0.000000,0.002119,0.000000,0.008475,0.006356,0.012712,0.000000
5,475,2810,37,4.692632,2.247396,12.837838,6.171046,12.617865,0.452632,0.069474,...,0.006316,0.006316,0.000000,0.002105,0.000000,0.002105,0.000000,0.004211,0.000000,0.000000
6,459,2620,35,4.442266,2.028780,13.114286,6.181869,12.555845,0.455338,0.074074,...,0.019608,0.004357,0.002179,0.002179,0.000000,0.000000,0.000000,0.004357,0.004357,0.002179
7,442,2985,17,5.538462,3.020985,26.000000,10.035232,13.365807,0.504525,0.076923,...,0.000000,0.011312,0.002262,0.000000,0.002262,0.000000,0.000000,0.002262,0.002262,0.000000
8,372,2615,16,5.768817,2.962180,23.250000,8.649422,13.272982,0.553763,0.080645,...,0.013441,0.002688,0.002688,0.002688,0.000000,0.002688,0.000000,0.002688,0.000000,0.000000
9,368,2226,19,4.896739,2.624133,19.368421,7.065189,12.458736,0.524457,0.065217,...,0.002717,0.000000,0.000000,0.000000,0.000000,0.005435,0.000000,0.008152,0.002717,0.000000


In [14]:
test_heavy_300.to_csv(
    r"C:\Users\Jamie\Desktop\DSM500 Final Project\Code\Final datasets v2 last update\test_heavy_300_features.csv",
    index = False
)

___

In [15]:
# all copied from above
train_df = pd.read_csv(
    r"C:\Users\Jamie\Desktop\DSM500 Final Project\Code\Final datasets v2 last update\test_light_300.csv"
)

# run surface feature extraction across entire test set
test_light_300 = pd.DataFrame(
    train_df["text"].apply(extract_surface_features
    # and convert to list of dicts
    ).to_list()
)

# check output
print(test_light_300.shape)
print(test_light_300.isna().sum().sum())
print(test_light_300.dtypes.value_counts())
test_light_300.head(10)

(300, 96)
0
float64    93
int64       3
Name: count, dtype: int64


,word_count,character_count,sentence_count,mean_word_length,std_word_length,mean_sentence_length,std_sentence_length,root_ttr,hapax_l_ratio,hapax_d_ratio,...,func_ratio_was,func_ratio_were,func_ratio_what,func_ratio_when,func_ratio_which,func_ratio_who,func_ratio_will,func_ratio_with,func_ratio_would,func_ratio_your
0,379,2407,16,5.092348,2.482534,23.687500,8.060077,11.762940,0.467018,0.068602,...,0.000000,0.002639,0.000000,0.005277,0.002639,0.002639,0.010554,0.002639,0.005277,0.0
1,402,2675,21,5.144279,2.647458,19.142857,7.099871,8.628456,0.256219,0.089552,...,0.019900,0.002488,0.000000,0.000000,0.000000,0.000000,0.000000,0.009950,0.000000,0.0
2,337,1976,16,4.341246,2.399157,21.062500,12.059585,11.003641,0.456973,0.068249,...,0.005935,0.002967,0.000000,0.002967,0.000000,0.002967,0.000000,0.008902,0.005935,0.0
3,417,3166,22,6.148681,2.969381,18.954545,6.048871,12.879165,0.472422,0.091127,...,0.000000,0.000000,0.000000,0.002398,0.000000,0.000000,0.000000,0.004796,0.000000,0.0
4,390,2789,17,5.851282,2.929678,22.941176,5.285614,12.254146,0.489744,0.064103,...,0.000000,0.000000,0.000000,0.000000,0.002564,0.000000,0.007692,0.005128,0.012821,0.0
5,403,2311,33,4.545906,2.182664,12.212121,5.876288,11.307675,0.411911,0.096774,...,0.009926,0.002481,0.000000,0.000000,0.000000,0.002481,0.000000,0.004963,0.000000,0.0
6,422,2358,29,4.284360,1.937334,14.551724,8.783159,11.293583,0.409953,0.073460,...,0.021327,0.000000,0.002370,0.004739,0.000000,0.000000,0.000000,0.009479,0.002370,0.0
7,378,2424,13,5.198413,2.973480,29.076923,12.676214,11.984227,0.476190,0.074074,...,0.005291,0.007937,0.005291,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0
8,385,2601,15,5.545455,2.976730,25.666667,15.399856,12.333462,0.475325,0.093506,...,0.020779,0.002597,0.002597,0.002597,0.000000,0.002597,0.000000,0.007792,0.002597,0.0
9,355,2046,18,4.628169,2.549188,19.722222,6.714872,11.570239,0.495775,0.053521,...,0.000000,0.000000,0.000000,0.002817,0.000000,0.002817,0.000000,0.008451,0.002817,0.0


In [16]:
test_light_300.to_csv(
    r"C:\Users\Jamie\Desktop\DSM500 Final Project\Code\Final datasets v2 last update\test_light_300_features.csv",
    index = False
)

___

In [17]:
# all copied from above
train_df = pd.read_csv(
    r"C:\Users\Jamie\Desktop\DSM500 Final Project\Code\Final datasets v2 last update\test_original_300.csv"
)

# run surface feature extraction across entire test set
test_original_300 = pd.DataFrame(
    train_df["text"].apply(extract_surface_features
    # and convert to list of dicts
    ).to_list()
)

# check output
print(test_original_300.shape)
print(test_original_300.isna().sum().sum())
print(test_original_300.dtypes.value_counts())
test_original_300.head(10)

(300, 96)
0
float64    93
int64       3
Name: count, dtype: int64


,word_count,character_count,sentence_count,mean_word_length,std_word_length,mean_sentence_length,std_sentence_length,root_ttr,hapax_l_ratio,hapax_d_ratio,...,func_ratio_was,func_ratio_were,func_ratio_what,func_ratio_when,func_ratio_which,func_ratio_who,func_ratio_will,func_ratio_with,func_ratio_would,func_ratio_your
0,378,2363,16,4.997354,2.365424,23.625000,6.918047,11.572751,0.455026,0.066138,...,0.002646,0.002646,0.000000,0.005291,0.002646,0.002646,0.010582,0.002646,0.005291,0.0
1,384,2575,20,5.080729,2.563007,19.200000,7.379702,8.164966,0.250000,0.072917,...,0.026042,0.002604,0.000000,0.002604,0.000000,0.000000,0.000000,0.010417,0.000000,0.0
2,335,1909,16,4.220896,2.284048,20.937500,11.723207,10.544716,0.423881,0.068657,...,0.005970,0.005970,0.000000,0.002985,0.000000,0.002985,0.000000,0.008955,0.005970,0.0
3,414,3160,22,6.176329,2.997635,18.818182,5.820312,12.532566,0.451691,0.099034,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.004831,0.000000,0.0
4,393,2741,17,5.653944,2.880546,23.117647,5.592877,12.106399,0.475827,0.071247,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.005089,0.007634,0.012723,0.0
5,396,2254,33,4.512626,2.139655,12.000000,5.752470,11.306675,0.419192,0.093434,...,0.010101,0.000000,0.000000,0.000000,0.000000,0.002525,0.000000,0.002525,0.000000,0.0
6,429,2353,29,4.209790,1.821344,14.793103,9.026319,11.152785,0.391608,0.076923,...,0.016317,0.004662,0.002331,0.004662,0.002331,0.000000,0.000000,0.009324,0.002331,0.0
7,368,2358,13,5.182065,2.908384,28.307692,12.880424,12.302350,0.508152,0.065217,...,0.008152,0.008152,0.005435,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0
8,391,2523,15,5.276215,2.806719,26.066667,14.959798,12.137322,0.465473,0.092072,...,0.020460,0.002558,0.002558,0.000000,0.000000,0.002558,0.000000,0.007673,0.000000,0.0
9,358,2005,18,4.463687,2.401715,19.888889,6.814110,11.468806,0.472067,0.075419,...,0.002793,0.000000,0.002793,0.002793,0.002793,0.002793,0.000000,0.008380,0.002793,0.0


In [18]:
test_original_300.to_csv(
    r"C:\Users\Jamie\Desktop\DSM500 Final Project\Code\Final datasets v2 last update\test_original_300_features.csv",
    index = False
)

___

In [19]:
# all copied from above
train_df = pd.read_csv(
    r"C:\Users\Jamie\Desktop\DSM500 Final Project\Code\Final datasets v2 last update\test_trans_300.csv"
)

# run surface feature extraction across entire test set
test_trans_300 = pd.DataFrame(
    train_df["text"].apply(extract_surface_features
    # and convert to list of dicts
    ).to_list()
)

# check output
print(test_trans_300.shape)
print(test_trans_300.isna().sum().sum())
print(test_trans_300.dtypes.value_counts())
test_trans_300.head(10)

(300, 96)
0
float64    93
int64       3
Name: count, dtype: int64


,word_count,character_count,sentence_count,mean_word_length,std_word_length,mean_sentence_length,std_sentence_length,root_ttr,hapax_l_ratio,hapax_d_ratio,...,func_ratio_was,func_ratio_were,func_ratio_what,func_ratio_when,func_ratio_which,func_ratio_who,func_ratio_will,func_ratio_with,func_ratio_would,func_ratio_your
0,389,2411,16,4.915167,2.408319,24.312500,7.727538,11.610761,0.449871,0.071979,...,0.000000,0.002571,0.000000,0.005141,0.002571,0.005141,0.010283,0.002571,0.005141,0.0
1,403,2700,21,5.109181,2.605204,19.190476,7.737473,7.820727,0.215881,0.076923,...,0.024814,0.000000,0.000000,0.002481,0.000000,0.000000,0.000000,0.014888,0.000000,0.0
2,346,1942,15,4.248555,2.338806,23.066667,10.871778,11.020868,0.453757,0.069364,...,0.005780,0.005780,0.000000,0.002890,0.000000,0.002890,0.000000,0.005780,0.002890,0.0
3,435,3246,21,6.089655,2.987139,20.714286,7.850647,12.418100,0.439080,0.089655,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.004598,0.000000,0.0
4,401,2798,17,5.688279,2.843391,23.588235,5.831545,12.384529,0.483791,0.064838,...,0.000000,0.000000,0.000000,0.000000,0.002494,0.000000,0.007481,0.007481,0.012469,0.0
5,395,2309,33,4.655696,2.288759,11.969697,5.702125,11.421610,0.430380,0.091139,...,0.010127,0.000000,0.000000,0.002532,0.000000,0.002532,0.000000,0.005063,0.000000,0.0
6,433,2386,29,4.212471,1.893500,14.931034,9.142289,10.908924,0.376443,0.073903,...,0.016166,0.004619,0.002309,0.006928,0.002309,0.000000,0.000000,0.009238,0.004619,0.0
7,397,2540,13,5.173804,2.914831,30.538462,13.697925,11.995066,0.468514,0.065491,...,0.007557,0.005038,0.007557,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0
8,426,2705,15,5.211268,2.852799,28.400000,16.495656,11.821839,0.410798,0.093897,...,0.016432,0.002347,0.002347,0.000000,0.000000,0.002347,0.000000,0.004695,0.002347,0.0
9,371,2083,18,4.479784,2.504227,20.611111,6.758853,11.577583,0.463612,0.078167,...,0.000000,0.000000,0.000000,0.002695,0.005391,0.002695,0.000000,0.010782,0.000000,0.0


In [20]:
test_trans_300.to_csv(
    r"C:\Users\Jamie\Desktop\DSM500 Final Project\Code\Final datasets v2 last update\test_trans_300_features.csv",
    index = False
)